In [48]:
from qdrant_client import QdrantClient
import os
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext, Settings, SimpleDirectoryReader, SummaryIndex
from llama_index.core.schema import TextNode
from llama_index.embeddings.openai import OpenAIEmbedding
from dotenv import load_dotenv
from qdrant_client.http.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)
import json
import logging
import time
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    granite_picture_description
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem 
from docling.chunking import HybridChunker, HierarchicalChunker
from docling.datamodel.document import DoclingDocument
from llama_index.llms.openai_like import OpenAILike
#qua fatto porcata per inserire utils
import importlib
import utils as utils
utils = importlib.reload(utils)
from utils import iniezionetagimmagini, riassuntodocumento, metadata_extraction

load_dotenv()

True

In [ ]:
collectionname = "WAMASSMALLTOBIGWINDOW"

url_embedder = os.getenv("VLLM_API_BASE_URL")

url_qdrant = os.getenv("QDRANT_URL")

client = qdrant_client.QdrantClient(url=url_qdrant)

embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)

Settings.embed_model = embed_model


vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)


2025-12-16 11:21:18,627 - INFO - HTTP Request: GET http://10.1.1.193:6333 "HTTP/1.1 200 OK"
2025-12-16 11:21:18,646 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW/exists "HTTP/1.1 200 OK"
2025-12-16 11:21:18,660 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW "HTTP/1.1 200 OK"
2025-12-16 11:21:18,664 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW/exists "HTTP/1.1 200 OK"
2025-12-16 11:21:18,668 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW "HTTP/1.1 200 OK"
2025-12-16 11:21:18,726 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW/exists "HTTP/1.1 200 OK"
2025-12-16 11:21:18,729 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW "HTTP/1.1 200 OK"


In [ ]:
if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(
            size=1024,   
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

2025-12-16 11:21:18,833 - INFO - HTTP Request: GET http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW/exists "HTTP/1.1 200 OK"
2025-12-16 11:21:18,892 - INFO - HTTP Request: DELETE http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW "HTTP/1.1 200 OK"
2025-12-16 11:21:19,064 - INFO - HTTP Request: PUT http://10.1.1.193:6333/collections/WAMASSMALLTOBIGWINDOW "HTTP/1.1 200 OK"


True

In [ ]:
import sqlite3


conn = sqlite3.connect("../knowledge_base.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS document_chunks (
        filename TEXT,
        chunk_index INTEGER,
        text_content TEXT,
        PRIMARY KEY (filename, chunk_index)
    )
""")
conn.commit()

In [ ]:


def smalltobig(DOC_SOURCE, DOC_SOURCE_MD):
    doc = DoclingDocument.load_from_json(DOC_SOURCE)
    doc = iniezionetagimmagini(doc)
    
    nodes = []
    chunker = HybridChunker()
    chunks = list(chunker.chunk(doc))
    
    
    sql_data = []
    
    for i, chunk in enumerate(chunks):
    
        small_text = chunker.contextualize(chunk=chunk)
        clean_meta = metadata_extraction(chunk)
        filename = clean_meta["origin_filename"]
        

        sql_data.append((filename, i, small_text))
        

        clean_meta["chunk_index"] = i  
        clean_meta["chunk_id"] = f"{i}:{filename}"
        
        new_node = TextNode(
            text=small_text,
            metadata=clean_meta
        )
        nodes.append(new_node)
        
    
    cursor.executemany(
        "INSERT OR REPLACE INTO document_chunks (filename, chunk_index, text_content) VALUES (?, ?, ?)", 
        sql_data
    )
    conn.commit()

    
    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        show_progress=True
    )
    print(f"Documento {DOC_SOURCE} processato. Chunks salvati in SQL e Qdrant.")


In [ ]:
base_folder = "../preprocessing/scratch"


json_file = ""
md_file = ""


for root, dirs, files in os.walk(base_folder):
    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)
    if json_file and md_file:
        #print(json_file)
        smalltobig(DOC_SOURCE=json_file, DOC_SOURCE_MD=md_file)

